In [ ]:
# 1) Setup: check GPU, install deps, mount Drive (checkpoints are saved there so you can resume)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q tiktoken datasets
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Download + tokenize data (FineWeb-Edu) -> /content/data/train.bin  (run ONCE, ~1-2 hours for 3B tokens)
import os, numpy as np, tiktoken
from datasets import load_dataset

TARGET_TOKENS = 3_000_000_000      # ~20 tokens/param for a 150M model. Lower it (e.g. 500_000_000) for a quick test run.
DATA_DIR = "/content/data"
DRIVE_DIR = "/content/drive/MyDrive/gpt150m"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(DRIVE_DIR, exist_ok=True)
BIN = f"{DATA_DIR}/train.bin"

if os.path.exists(f"{DRIVE_DIR}/train.bin") and not os.path.exists(BIN):
    print("Found tokenized data on Drive, copying..."); os.system(f"cp {DRIVE_DIR}/train.bin {BIN}")

if not os.path.exists(BIN):
    enc = tiktoken.get_encoding("gpt2"); EOT = enc.eot_token
    ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
    out = np.memmap(BIN, dtype=np.uint16, mode="w+", shape=(TARGET_TOKENS,))
    n, buf = 0, []
    def flush(buf, n):
        ids = enc.encode_ordinary_batch(buf, num_threads=os.cpu_count())
        flat = np.fromiter((t for d in ids for t in (*d, EOT)), dtype=np.uint16)
        k = min(len(flat), TARGET_TOKENS - n)
        out[n:n+k] = flat[:k]
        return n + k
    for ex in ds:
        buf.append(ex["text"])
        if len(buf) == 2000:
            n = flush(buf, n); buf = []
            if (n // 2000) % 50 == 0: print(f"{n/1e6:.0f}M tokens", end="\r")
            if n >= TARGET_TOKENS: break
    if buf and n < TARGET_TOKENS: n = flush(buf, n)
    out.flush()
    np.save(f"{DATA_DIR}/n_tokens.npy", np.array([n]))
    print(f"\nDone: {n/1e6:.0f}M tokens")
    os.system(f"cp {BIN} {DRIVE_DIR}/train.bin; cp {DATA_DIR}/n_tokens.npy {DRIVE_DIR}/n_tokens.npy")  # backup to Drive


In [ ]:
# 3) Model: ~152M-param GPT (RoPE, RMSNorm, tied embeddings, flash attention via SDPA)
import math, time, os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 50304      # GPT-2 vocab (50257) padded to a multiple of 64
    block_size: int = 1024
    n_layer: int = 16
    n_head: int = 12
    n_embd: int = 768

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x) * self.weight

def rope_cache(T, D, device, base=10000.0):
    inv = 1.0 / (base ** (torch.arange(0, D, 2, device=device).float() / D))
    f = torch.outer(torch.arange(T, device=device).float(), inv)
    return f.cos(), f.sin()

def apply_rope(x, cos, sin):      # x: (B, H, T, D)
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

class Attention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.h = c.n_head
        self.qkv = nn.Linear(c.n_embd, 3 * c.n_embd, bias=False)
        self.proj = nn.Linear(c.n_embd, c.n_embd, bias=False)
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, C // self.h).permute(2, 0, 3, 1, 4)
        q, k = apply_rope(q, cos[:T], sin[:T]), apply_rope(k, cos[:T], sin[:T])
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class MLP(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.fc = nn.Linear(c.n_embd, 4 * c.n_embd, bias=False)
        self.proj = nn.Linear(4 * c.n_embd, c.n_embd, bias=False)
    def forward(self, x): return self.proj(F.gelu(self.fc(x), approximate="tanh"))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n1, self.attn, self.n2, self.mlp = RMSNorm(c.n_embd), Attention(c), RMSNorm(c.n_embd), MLP(c)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.n1(x), cos, sin)
        return x + self.mlp(self.n2(x))

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.wte = nn.Embedding(c.vocab_size, c.n_embd)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.nf = RMSNorm(c.n_embd)
        self.lm_head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight           # weight tying
        self.apply(self._init)
        for n, p in self.named_parameters():            # scaled init for residual projections
            if n.endswith("proj.weight"): nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * c.n_layer))
        cos, sin = rope_cache(c.block_size, c.n_embd // c.n_head, "cpu")
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None):
        x = self.wte(idx)
        for b in self.blocks: x = b(x, self.cos, self.sin)
        logits = self.lm_head(self.nf(x))
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)).float(), targets.view(-1)) if targets is not None else None
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new=200, temperature=0.8, top_k=50):
        for _ in range(max_new):
            logits, _ = self(idx[:, -self.c.block_size:])
            logits = logits[:, -1, :50257] / temperature          # ignore padded vocab entries
            v, _ = torch.topk(logits, top_k); logits[logits < v[:, [-1]]] = -float("inf")
            idx = torch.cat([idx, torch.multinomial(F.softmax(logits, -1), 1)], dim=1)
        return idx

cfg = Config()
_m = GPT(cfg)
print(f"Parameters: {sum(p.numel() for p in _m.parameters())/1e6:.1f}M"); del _m


In [ ]:
import os
os.makedirs("/content/data", exist_ok=True)
!cp /content/drive/MyDrive/gpt150m/n_tokens.npy /content/data/
!ls -lh /content/data /content/drive/MyDrive/gpt150m

In [ ]:
# 4) Train (auto-resumes from Drive checkpoint if the session dies: just re-run cells 1, 3, 4)
DATA_DIR = "/content/data"; DRIVE_DIR = "/content/drive/MyDrive/gpt150m"; CKPT = f"{DRIVE_DIR}/ckpt.pt"
if not os.path.exists(f"{DATA_DIR}/train.bin"):
    os.makedirs(DATA_DIR, exist_ok=True)
    os.system(f"cp {DRIVE_DIR}/train.bin {DATA_DIR}/train.bin; cp {DRIVE_DIR}/n_tokens.npy {DATA_DIR}/n_tokens.npy")

# --- hyperparameters ---
MICRO_BS, GRAD_ACCUM = 32, 16                        # 32*1024*16 = 524,288 tokens per optimizer step
MAX_LR, MIN_LR, WARMUP = 6e-4, 6e-5, 200
WEIGHT_DECAY, CLIP = 0.1, 1.0
EVAL_EVERY, EVAL_ITERS, SAVE_EVERY, VAL_TOKENS = 250, 20, 250, 10_000_000

n_tok = int(np.load(f"{DATA_DIR}/n_tokens.npy")[0])
data = np.memmap(f"{DATA_DIR}/train.bin", dtype=np.uint16, mode="r", shape=(n_tok,))
train_data, val_data = data[:-VAL_TOKENS], data[-VAL_TOKENS:]
TOK_PER_STEP = MICRO_BS * cfg.block_size * GRAD_ACCUM
MAX_STEPS = len(train_data) // TOK_PER_STEP          # exactly one pass over the data
print(f"{len(train_data)/1e9:.2f}B train tokens -> {MAX_STEPS} steps")

device = "cuda"
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
raw_model = GPT(cfg).to(device)
model = torch.compile(raw_model)

decay = [p for n, p in raw_model.named_parameters() if p.dim() >= 2]
nodecay = [p for n, p in raw_model.named_parameters() if p.dim() < 2]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": nodecay, "weight_decay": 0.0}],
                        lr=MAX_LR, betas=(0.9, 0.95), fused=True)

step = 0
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device)
    raw_model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); step = ck["step"]
    print(f"Resumed from step {step}")

def get_lr(s):
    if s < WARMUP: return MAX_LR * (s + 1) / WARMUP
    r = (s - WARMUP) / max(1, MAX_STEPS - WARMUP)
    return MIN_LR + 0.5 * (MAX_LR - MIN_LR) * (1 + math.cos(math.pi * r))

def get_batch(src):
    ix = torch.randint(len(src) - cfg.block_size - 1, (MICRO_BS,)).numpy()
    x = np.stack([src[i : i + cfg.block_size] for i in ix]).astype(np.int64)
    y = np.stack([src[i + 1 : i + 1 + cfg.block_size] for i in ix]).astype(np.int64)
    return torch.from_numpy(x).to(device, non_blocking=True), torch.from_numpy(y).to(device, non_blocking=True)

@torch.no_grad()
def eval_loss():
    model.eval(); tot = 0.0
    for _ in range(EVAL_ITERS):
        x, y = get_batch(val_data)
        with torch.autocast("cuda", dtype=torch.bfloat16): tot += model(x, y)[1].item()
    model.train(); return tot / EVAL_ITERS

def save():
    torch.save({"model": raw_model.state_dict(), "opt": opt.state_dict(), "step": step, "cfg": cfg.__dict__}, CKPT + ".tmp")
    os.replace(CKPT + ".tmp", CKPT)

model.train(); t0 = time.time()
while step < MAX_STEPS:
    for g in opt.param_groups: g["lr"] = get_lr(step)
    loss_acc = 0.0
    for _ in range(GRAD_ACCUM):
        x, y = get_batch(train_data)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            _, loss = model(x, y)
        (loss / GRAD_ACCUM).backward(); loss_acc += loss.item() / GRAD_ACCUM
    norm = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), CLIP)
    opt.step(); opt.zero_grad(set_to_none=True); step += 1
    if step % 10 == 0:
        dt = time.time() - t0; t0 = time.time()
        print(f"step {step}/{MAX_STEPS} | loss {loss_acc:.3f} | lr {get_lr(step):.2e} | gnorm {norm:.2f} | {10*TOK_PER_STEP/dt/1e3:.0f}k tok/s")
    if step % EVAL_EVERY == 0: print(f"  >>> val loss {eval_loss():.3f}")
    if step % SAVE_EVERY == 0: save()
save(); print("Training done.")


In [ ]:
# 5) Generate text from the trained model
import tiktoken
enc = tiktoken.get_encoding("gpt2")
raw_model.eval()
prompt = "The history of the Roman Empire"
idx = torch.tensor([enc.encode_ordinary(prompt)], device=device)
with torch.autocast("cuda", dtype=torch.bfloat16):
    out = raw_model.generate(idx, max_new=200)
print(enc.decode(out[0].tolist()))

In [ ]:
!cp /content/drive/MyDrive/gpt150m/ckpt.pt /content/drive/MyDrive/gpt150m/ckpt_3B_final.pt

In [ ]:
import torch, torch.nn.functional as F

def ask(question, max_new=60, temperature=0.7, top_k=40, rep_penalty=1.1):
    prompt = (
        "Q: What is the capital of France?\nA: The capital of France is Paris.\n\n"
        "Q: What do plants need to grow?\nA: Plants need sunlight, water, and nutrients from the soil.\n\n"
        "Q: Who wrote the play Romeo and Juliet?\nA: Romeo and Juliet was written by William Shakespeare.\n\n"
        f"Q: {question}\nA:"
    )
    idx = torch.tensor([enc.encode_ordinary(prompt)], device=device)
    start = idx.shape[1]
    raw_model.eval()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        for _ in range(max_new):
            logits, _ = raw_model(idx[:, -raw_model.c.block_size:])
            logits = logits[:, -1, :50257].float()
            seen = torch.unique(idx[0, -128:])
            l = logits[0, seen]
            logits[0, seen] = torch.where(l > 0, l / rep_penalty, l * rep_penalty)
            logits = logits / temperature
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")
            idx = torch.cat([idx, torch.multinomial(F.softmax(logits, -1), 1)], dim=1)
            text = enc.decode(idx[0, start:].tolist())
            if "\n" in text.strip("\n ") : break      # stop at the end of the answer line
    return enc.decode(idx[0, start:].tolist()).strip().split("\n")[0]

print(ask("What is photosynthesis?"))
print(ask("Where does the sun rise?"))

In [ ]:
# STEP 1) Setup + back up your finished model (the continued run never touches this backup)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q tiktoken datasets
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_DIR = "/content/drive/MyDrive/gpt150m"
src, dst = f"{DRIVE_DIR}/ckpt.pt", f"{DRIVE_DIR}/ckpt_3B_final.pt"
if os.path.exists(src) and not os.path.exists(dst):
    shutil.copy(src, dst); print("Backed up ckpt.pt -> ckpt_3B_final.pt")
print(os.listdir(DRIVE_DIR))

In [ ]:
# STEP 2) Tokenize NEW data (documents the first run never saw) -> /content/data/train2.bin
import os, numpy as np, tiktoken
from datasets import load_dataset

NEW_TOKENS = 1_000_000_000     # 1B = ~75 min of training. Use 800_000_000 for ~1 hour.
SKIP_DOCS = 3_100_000          # skip the documents already used in run 1 (~3B tokens, ~1,000 tokens/doc)
DATA_DIR = "/content/data"; DRIVE_DIR = "/content/drive/MyDrive/gpt150m"
os.makedirs(DATA_DIR, exist_ok=True)
BIN2 = f"{DATA_DIR}/train2.bin"

# the ORIGINAL data file is still needed: its last 10M tokens are the held-out validation set
for f in ("train.bin", "n_tokens.npy"):
    if not os.path.exists(f"{DATA_DIR}/{f}"):
        print("copying", f, "from Drive..."); os.system(f"cp {DRIVE_DIR}/{f} {DATA_DIR}/{f}")
for f in ("train2.bin", "n_tokens2.npy"):
    if os.path.exists(f"{DRIVE_DIR}/{f}") and not os.path.exists(f"{DATA_DIR}/{f}"):
        print("copying", f, "from Drive..."); os.system(f"cp {DRIVE_DIR}/{f} {DATA_DIR}/{f}")

if not os.path.exists(BIN2):
    enc = tiktoken.get_encoding("gpt2"); EOT = enc.eot_token
    ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True).skip(SKIP_DOCS)
    out = np.memmap(BIN2, dtype=np.uint16, mode="w+", shape=(NEW_TOKENS,))
    n, buf, last = 0, [], 0
    def flush(buf, n):
        ids = enc.encode_ordinary_batch(buf, num_threads=os.cpu_count())
        flat = np.fromiter((t for d in ids for t in (*d, EOT)), dtype=np.uint16)
        k = min(len(flat), NEW_TOKENS - n)
        out[n:n+k] = flat[:k]
        return n + k
    for ex in ds:
        buf.append(ex["text"])
        if len(buf) == 5000:
            n = flush(buf, n); buf = []
            if n - last >= 100_000_000: print(f"{n/1e6:.0f}M / {NEW_TOKENS/1e6:.0f}M tokens"); last = n
            if n >= NEW_TOKENS: break
    if buf and n < NEW_TOKENS: n = flush(buf, n)
    out.flush()
    np.save(f"{DATA_DIR}/n_tokens2.npy", np.array([n]))
    print(f"Done: {n/1e6:.0f}M tokens")
    os.system(f"cp {BIN2} {DRIVE_DIR}/train2.bin; cp {DATA_DIR}/n_tokens2.npy {DRIVE_DIR}/n_tokens2.npy")
else:
    print("train2.bin already exists, skipping")


In [ ]:
# STEP 3) Model (identical to before): ~152M-param GPT (RoPE, RMSNorm, tied embeddings, flash attention via SDPA)
import math, time, os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 50304      # GPT-2 vocab (50257) padded to a multiple of 64
    block_size: int = 1024
    n_layer: int = 16
    n_head: int = 12
    n_embd: int = 768

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x) * self.weight

def rope_cache(T, D, device, base=10000.0):
    inv = 1.0 / (base ** (torch.arange(0, D, 2, device=device).float() / D))
    f = torch.outer(torch.arange(T, device=device).float(), inv)
    return f.cos(), f.sin()

def apply_rope(x, cos, sin):      # x: (B, H, T, D)
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

class Attention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.h = c.n_head
        self.qkv = nn.Linear(c.n_embd, 3 * c.n_embd, bias=False)
        self.proj = nn.Linear(c.n_embd, c.n_embd, bias=False)
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, C // self.h).permute(2, 0, 3, 1, 4)
        q, k = apply_rope(q, cos[:T], sin[:T]), apply_rope(k, cos[:T], sin[:T])
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class MLP(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.fc = nn.Linear(c.n_embd, 4 * c.n_embd, bias=False)
        self.proj = nn.Linear(4 * c.n_embd, c.n_embd, bias=False)
    def forward(self, x): return self.proj(F.gelu(self.fc(x), approximate="tanh"))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n1, self.attn, self.n2, self.mlp = RMSNorm(c.n_embd), Attention(c), RMSNorm(c.n_embd), MLP(c)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.n1(x), cos, sin)
        return x + self.mlp(self.n2(x))

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.wte = nn.Embedding(c.vocab_size, c.n_embd)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.nf = RMSNorm(c.n_embd)
        self.lm_head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight           # weight tying
        self.apply(self._init)
        for n, p in self.named_parameters():            # scaled init for residual projections
            if n.endswith("proj.weight"): nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * c.n_layer))
        cos, sin = rope_cache(c.block_size, c.n_embd // c.n_head, "cpu")
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None):
        x = self.wte(idx)
        for b in self.blocks: x = b(x, self.cos, self.sin)
        logits = self.lm_head(self.nf(x))
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)).float(), targets.view(-1)) if targets is not None else None
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new=200, temperature=0.8, top_k=50):
        for _ in range(max_new):
            logits, _ = self(idx[:, -self.c.block_size:])
            logits = logits[:, -1, :50257] / temperature          # ignore padded vocab entries
            v, _ = torch.topk(logits, top_k); logits[logits < v[:, [-1]]] = -float("inf")
            idx = torch.cat([idx, torch.multinomial(F.softmax(logits, -1), 1)], dim=1)
        return idx

cfg = Config()
_m = GPT(cfg)
print(f"Parameters: {sum(p.numel() for p in _m.parameters())/1e6:.1f}M"); del _m

In [ ]:
# STEP 4) Continue training on the new tokens (auto-resumes from ckpt_ext.pt if the session dies: re-run steps 1, 3, 4)
DATA_DIR = "/content/data"; DRIVE_DIR = "/content/drive/MyDrive/gpt150m"
CKPT_BASE = f"{DRIVE_DIR}/ckpt_3B_final.pt"     # finished run-1 model (read only)
CKPT_EXT = f"{DRIVE_DIR}/ckpt_ext.pt"           # continued run saves here

MICRO_BS, GRAD_ACCUM = 32, 16                   # 524,288 tokens per step (same as before)
MAX_LR, MIN_LR, WARMUP = 3e-4, 3e-5, 50         # lower peak LR than run 1, short warmup, cosine decay again
WEIGHT_DECAY, CLIP = 0.1, 1.0
EVAL_EVERY, EVAL_ITERS, SAVE_EVERY, VAL_TOKENS = 250, 20, 250, 10_000_000

n_old = int(np.load(f"{DATA_DIR}/n_tokens.npy")[0])
val_data = np.memmap(f"{DATA_DIR}/train.bin", dtype=np.uint16, mode="r", shape=(n_old,))[-VAL_TOKENS:]   # same val set as run 1
n_new = int(np.load(f"{DATA_DIR}/n_tokens2.npy")[0])
train_data = np.memmap(f"{DATA_DIR}/train2.bin", dtype=np.uint16, mode="r", shape=(n_new,))
TOK_PER_STEP = MICRO_BS * cfg.block_size * GRAD_ACCUM
MAX_STEPS = len(train_data) // TOK_PER_STEP
print(f"{len(train_data)/1e9:.2f}B NEW train tokens -> {MAX_STEPS} steps (~{MAX_STEPS*TOK_PER_STEP/221e3/60:.0f} min)")

device = "cuda"
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
raw_model = GPT(cfg).to(device)
model = torch.compile(raw_model)
decay = [p for n, p in raw_model.named_parameters() if p.dim() >= 2]
nodecay = [p for n, p in raw_model.named_parameters() if p.dim() < 2]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": nodecay, "weight_decay": 0.0}],
                        lr=MAX_LR, betas=(0.9, 0.95), fused=True)

step = 0
if os.path.exists(CKPT_EXT):
    ck = torch.load(CKPT_EXT, map_location=device)
    raw_model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); step = ck["step"]
    print(f"Resumed continued run at step {step}")
else:
    ck = torch.load(CKPT_BASE, map_location=device)
    raw_model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    print("Loaded run-1 model; starting new schedule at step 0")
del ck

def get_lr(s):
    if s < WARMUP: return MAX_LR * (s + 1) / WARMUP
    r = (s - WARMUP) / max(1, MAX_STEPS - WARMUP)
    return MIN_LR + 0.5 * (MAX_LR - MIN_LR) * (1 + math.cos(math.pi * r))

def get_batch(src):
    ix = torch.randint(len(src) - cfg.block_size - 1, (MICRO_BS,)).numpy()
    x = np.stack([src[i : i + cfg.block_size] for i in ix]).astype(np.int64)
    y = np.stack([src[i + 1 : i + 1 + cfg.block_size] for i in ix]).astype(np.int64)
    return torch.from_numpy(x).to(device, non_blocking=True), torch.from_numpy(y).to(device, non_blocking=True)

@torch.no_grad()
def eval_loss():
    model.eval(); tot = 0.0
    for _ in range(EVAL_ITERS):
        x, y = get_batch(val_data)
        with torch.autocast("cuda", dtype=torch.bfloat16): tot += model(x, y)[1].item()
    model.train(); return tot / EVAL_ITERS

def save():
    torch.save({"model": raw_model.state_dict(), "opt": opt.state_dict(), "step": step, "cfg": cfg.__dict__}, CKPT_EXT + ".tmp")
    os.replace(CKPT_EXT + ".tmp", CKPT_EXT)

if step == 0: print(f"Starting val loss (run-1 model): {eval_loss():.3f}")
model.train(); t0 = time.time()
while step < MAX_STEPS:
    for g in opt.param_groups: g["lr"] = get_lr(step)
    loss_acc = 0.0
    for _ in range(GRAD_ACCUM):
        x, y = get_batch(train_data)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            _, loss = model(x, y)
        (loss / GRAD_ACCUM).backward(); loss_acc += loss.item() / GRAD_ACCUM
    norm = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), CLIP)
    opt.step(); opt.zero_grad(set_to_none=True); step += 1
    if step % 10 == 0:
        dt = time.time() - t0; t0 = time.time()
        print(f"step {step}/{MAX_STEPS} | loss {loss_acc:.3f} | lr {get_lr(step):.2e} | gnorm {norm:.2f} | {10*TOK_PER_STEP/dt/1e3:.0f}k tok/s")
    if step % EVAL_EVERY == 0: print(f"  >>> val loss {eval_loss():.3f}")
    if step % SAVE_EVERY == 0: save()
save(); print(f"Done. Final val loss {eval_loss():.3f}")


In [ ]:
# STEP 5) Try the model
import tiktoken, torch.nn.functional as F
enc = tiktoken.get_encoding("gpt2")
raw_model.eval()
prompt = "The Roman Empire was founded when"
idx = torch.tensor([enc.encode_ordinary(prompt)], device=device)
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    print(enc.decode(raw_model.generate(idx, max_new=150, temperature=0.8)[0].tolist()))


In [ ]:
print(f"{sum(p.numel() for p in raw_model.parameters())/1e6:.1f}M parameters")

In [ ]:
import copy, torch

def quant_dequant(w, bits):
    qmax = 2 ** (bits - 1) - 1
    scale = w.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / qmax
    return (w / scale).round().clamp(-qmax - 1, qmax) * scale

@torch.no_grad()
def val_loss(m, iters=40):
    torch.manual_seed(0); m.eval(); tot = 0.0       # same batches every time, so numbers compare fairly
    for _ in range(iters):
        x, y = get_batch(val_data)
        with torch.autocast("cuda", dtype=torch.bfloat16): tot += m(x, y)[1].item()
    return tot / iters

print(f"original (fp32 weights): {val_loss(raw_model):.4f}")
for bits in (8, 6, 4, 3):
    m = copy.deepcopy(raw_model)
    with torch.no_grad():
        for p in m.parameters():
            if p.dim() == 2: p.copy_(quant_dequant(p.float(), bits))   # weight matrices + embeddings, not norms
    print(f"int{bits}: {val_loss(m):.4f}")
    del m

In [ ]:
def quant_dequant_group(w, bits, group=64):
    o, i = w.shape
    wg = w.reshape(o, i // group, group)
    qmax = 2 ** (bits - 1) - 1
    scale = wg.abs().amax(dim=-1, keepdim=True).clamp(min=1e-8) / qmax
    return ((wg / scale).round().clamp(-qmax - 1, qmax) * scale).reshape(o, i)

for bits, g in [(4, 128), (4, 64), (4, 32), (3, 32)]:
    m = copy.deepcopy(raw_model)
    with torch.no_grad():
        for p in m.parameters():
            if p.dim() == 2: p.copy_(quant_dequant_group(p.float(), bits, g))
    print(f"int{bits} group {g}: {val_loss(m):.4f}")
    del m

In [ ]:
for emb_bits in (8, 6):
    m = copy.deepcopy(raw_model)
    with torch.no_grad():
        for n, p in m.named_parameters():
            if p.dim() != 2: continue
            if n == "wte.weight": p.copy_(quant_dequant(p.float(), emb_bits))
            else:                 p.copy_(quant_dequant_group(p.float(), 4, 64))
    print(f"blocks int4 g64 + embedding int{emb_bits}: {val_loss(m):.4f}")
    del m

In [ ]:
BLOCK, EMB = 113.2e6, 38.6e6          # weights in the 16 blocks vs the tied embedding

def run(bbits, group, ebits):
    m = copy.deepcopy(raw_model)
    with torch.no_grad():
        for n, p in m.named_parameters():
            if p.dim() != 2: continue
            if n == "wte.weight": p.copy_(quant_dequant(p.float(), ebits))
            elif group:           p.copy_(quant_dequant_group(p.float(), bbits, group))
            else:                 p.copy_(quant_dequant(p.float(), bbits))
    bpw = bbits + (16 / group if group else 0)           # bits per block weight incl. scales
    mb = (BLOCK * bpw + EMB * ebits) / 8 / 1e6
    print(f"blocks int{bbits} g{group or 'row'} + emb int{ebits}: loss {val_loss(m):.4f} | ~{mb:.0f} MB")
    del m

for cfg_ in [(6, None, 6), (6, 128, 6), (5, None, 8), (5, None, 6), (5, 128, 6), (5, 64, 6)]:
    run(*cfg_)

In [ ]:
# Export the model as a packed int5 (g64) + int6-embedding file, then verify it by reloading.
# Run in the SAME session that has raw_model, val_loss and copy (from the earlier quantization tests).
import os, copy, torch

import numpy as np

def unpack_bits(packed, n, bits):
    m = np.unpackbits(packed, bitorder="little", count=n * bits).reshape(n, bits)
    u = np.zeros(n, dtype=np.uint8)
    for j in range(bits): u |= (m[:, j] << j)
    return u

def dequantize(packed, scale, bits, rows, cols):
    g = cols // scale.shape[1]
    u = unpack_bits(packed, rows * cols, bits).reshape(rows, cols // g, g)
    q = u.astype(np.float32) - 2 ** (bits - 1)
    return (q * scale.astype(np.float32)[..., None]).reshape(rows, cols)

def load_weights(path):
    z = np.load(path)
    cfg, weights = [int(v) for v in z["config"]], {}
    for key in z.files:
        if key.endswith(".q"):
            name = key[:-2]
            bits, rows, cols = [int(v) for v in z[name + ".m"]]
            weights[name] = dequantize(z[key], z[name + ".s"], bits, rows, cols)
        elif key.endswith(".raw"):
            weights[key[:-4]] = z[key]
    return cfg, weights

def pack_bits(u, bits):
    u = u.astype(np.uint8).ravel()
    m = np.empty((u.size, bits), dtype=np.uint8)
    for j in range(bits): m[:, j] = (u >> j) & 1
    return np.packbits(m.ravel(), bitorder="little")

def quantize(w, bits, group):
    rows, cols = w.shape
    g = group if group else cols                       # group=None -> one scale per row
    wg = w.astype(np.float32).reshape(rows, cols // g, g)
    qmax = 2 ** (bits - 1) - 1
    scale = (np.abs(wg).max(-1, keepdims=True) / qmax).clip(min=1e-6).astype(np.float16)
    q = np.clip(np.rint(wg / scale.astype(np.float32)), -qmax - 1, qmax)
    u = (q + 2 ** (bits - 1)).astype(np.uint8)
    return pack_bits(u, bits), scale.reshape(rows, cols // g)

def export_model(model, path, block_bits=5, block_group=64, emb_bits=6):
    c = model.c
    out = {"config": np.array([c.vocab_size, c.block_size, c.n_layer, c.n_head, c.n_embd], dtype=np.int32)}
    for name, p in model.named_parameters():           # tied embedding appears once
        w = p.detach().float().cpu().numpy()
        if w.ndim == 2:
            bits, group = (emb_bits, None) if name == "wte.weight" else (block_bits, block_group)
            packed, scale = quantize(w, bits, group)
            out[name + ".q"], out[name + ".s"] = packed, scale
            out[name + ".m"] = np.array([bits, w.shape[0], w.shape[1]], dtype=np.int32)
        else:
            out[name + ".raw"] = w.astype(np.float32)
    np.savez(path, **out)

PATH = "/content/drive/MyDrive/gpt150m/gpt150m_int5g64.npz"
export_model(raw_model, PATH, block_bits=5, block_group=64, emb_bits=6)
print(f"saved {PATH}: {os.path.getsize(PATH)/1e6:.1f} MB")

# verify: rebuild weights from the FILE and measure validation loss
_, weights = load_weights(PATH)
m = copy.deepcopy(raw_model)
with torch.no_grad():
    for name, p in m.named_parameters():
        p.copy_(torch.from_numpy(weights[name]).to(p.device))
print(f"val loss from packed file: {val_loss(m):.4f}   (original: {val_loss(raw_model):.4f}, simulated int5 g64 test: 3.1839)")
del m

In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks"

In [ ]:
import json, re
from getpass import getpass

pat = re.compile(r"(ghp_|gho_|ghu_|ghs_|ghr_)[A-Za-z0-9]{20,}|github_pat_[A-Za-z0-9_]{20,}")
NB = "/content/drive/MyDrive/Colab Notebooks/YOUR_NOTEBOOK.ipynb"   # <- same path as before
TOKEN = getpass("GitHub token: ")

nb = json.load(open(NB))
hits = []
for i, c in enumerate(nb["cells"]):
    if pat.search("".join(c.get("source", []))): hits.append((i, "source"))
    if pat.search(json.dumps(c.get("outputs", []))): hits.append((i, "output"))
print("token found in (cell number, where):", hits)

for c in nb["cells"]:                                   # clear outputs, redact anything token-shaped in code
    if c["cell_type"] == "code":
        c["outputs"] = []; c["execution_count"] = None
    c["source"] = pat.sub("<REDACTED>", "".join(c.get("source", []))).splitlines(keepends=True)
clean = json.dumps(nb, indent=1)
assert not pat.search(clean), "a token is still in the file (maybe in metadata); don't push"

!rm -rf /content/slm
!git clone -q https://{TOKEN}@github.com/Murari17/slm.git /content/slm
open("/content/slm/train_gpt_150m.ipynb", "w").write(clean)
!cd /content/slm && git add . && git commit -q -m "Add training notebook (outputs cleared)" && git branch -M main && git push -u origin main